In [1]:
# Install required libraries
!pip install -q streamlit prophet plotly
print("✅ Dependencies installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 61.7 MB/s eta 0:00:00
✅ Dependencies installed successfully!


In [6]:
%%writefile bondmm_core.py
import math

"""
bondmm_core.py
Mathematical Core & Protocol Invariant Engine (Tran & Tran)
"""

def calculate_alpha(kappa: float) -> float:
    if kappa <= 0:
        raise ValueError("Kappa must be positive.")
    return kappa / (kappa + 1.0)

def calculate_K(r_star: float, T: float, alpha: float) -> float:
    if r_star <= 0 or T < 0:
        raise ValueError("r_star must be positive and T non-negative.")
    return r_star * math.pow(1 + T, alpha)

def get_reference_rate(x: float, y: float, r_star: float, kappa: float) -> float:
    """Implements virtual interest rate r = r_star * (x / y)^kappa"""
    if x <= 0 or y <= 0:
        raise ValueError("Reserves x and y must be strictly positive.")
    return r_star * math.pow(x / y, kappa)

def get_invariant_C(x: float, y: float, T: float, r_star: float, kappa: float) -> float:
    """Computes C = K * x^alpha + y^alpha"""
    alpha = calculate_alpha(kappa)
    K = calculate_K(r_star, T, alpha)
    return K * math.pow(x, alpha) + math.pow(y, alpha)

def execute_borrow(x: float, y: float, delta_x: float, T: float, r_star: float, kappa: float) -> dict:
    """
    Calculates post-trade state (x', y'), cash output delta_y,
    execution price, and slippage percentage.
    """
    if delta_x <= 0:
        raise ValueError("Delta x must be positive.")
    alpha = calculate_alpha(kappa)
    K = calculate_K(r_star, T, alpha)
    C = get_invariant_C(x, y, T, r_star, kappa)

    x_new = x + delta_x
    term = C - K * math.pow(x_new, alpha)
    if term <= 0:
        raise ValueError("Trade exceeds liquidity pool capacity.")

    y_new = math.pow(term, 1.0 / alpha)
    delta_y = y - y_new

    # Execution price (cash received per bond) and marginal price (initial spot)
    exec_price = delta_y / delta_x
    spot_price = y / (x * (1 + T))
    slippage_pct = max(0.0, ((spot_price - exec_price) / spot_price) * 100.0)

    return {
        "x_new": x_new,
        "y_new": y_new,
        "delta_y": delta_y,
        "exec_price": exec_price,
        "spot_price": spot_price,
        "slippage_pct": slippage_pct
    }

def execute_lend(x: float, y: float, delta_y: float, T: float, r_star: float, kappa: float) -> dict:
    """
    Calculates post-trade state (x', y'), bond output delta_x,
    execution price, and slippage percentage.
    """
    if delta_y <= 0:
        raise ValueError("Delta y must be positive.")
    alpha = calculate_alpha(kappa)
    K = calculate_K(r_star, T, alpha)
    C = get_invariant_C(x, y, T, r_star, kappa)

    y_new = y + delta_y
    term = C - math.pow(y_new, alpha)
    if term <= 0:
        raise ValueError("Trade exceeds liquidity pool capacity.")

    x_new = math.pow(term / K, 1.0 / alpha)
    delta_x = x - x_new

    exec_price = delta_y / delta_x
    spot_price = y / (x * (1 + T))
    slippage_pct = max(0.0, ((exec_price - spot_price) / spot_price) * 100.0)

    return {
        "x_new": x_new,
        "y_new": y_new,
        "delta_x": delta_x,
        "exec_price": exec_price,
        "spot_price": spot_price,
        "slippage_pct": slippage_pct
    }

def execute_cross_swap(x: float, y: float, T1: float, T2: float, amount: float, r_star: float, kappa: float) -> dict:
    """Single-transaction cross-maturity swap from T1 to T2"""
    borrow_res = execute_borrow(x, y, amount, T1, r_star, kappa)
    dy = borrow_res["delta_y"]
    lend_res = execute_lend(x, y, dy, T2, r_star, kappa)

    return {
        "x1_new": borrow_res["x_new"],
        "y1_new": borrow_res["y_new"],
        "x2_new": lend_res["x_new"],
        "y2_new": lend_res["y_new"],
        "intermediate_delta_y": dy,
        "final_delta_x2": lend_res["delta_x"]
    }

def check_lp_equity(x: float, y: float, T: float, min_ratio: float = 0.90) -> dict:
    """Measures LP Equity E = y + x/(1+T) against initial reference"""
    current_equity = y + (x / (1.0 + T))
    initial_equity = 1000.0 + (1000.0 / (1.0 + T))  # Base baseline
    ratio = current_equity / initial_equity
    return {
        "current_equity": current_equity,
        "ratio": ratio,
        "is_safe": ratio >= min_ratio
    }

# Baseline Pricing Formulas
def yield_space_borrow(x: float, y: float, delta_x: float, t: float, g: float = 0.1) -> dict:
    """Yield Protocol YieldSpace trade execution"""
    power = 1.0 - t
    k = (1.0 - g * t) * math.pow(x, power) + math.pow(y, power)
    x_new = x + delta_x
    term = k - (1.0 - g * t) * math.pow(x_new, power)
    if term <= 0:
        delta_y = y * 0.9  # Safety fallback cap
        y_new = y - delta_y
    else:
        y_new = math.pow(term, 1.0 / power)
        delta_y = y - y_new

    exec_price = delta_y / delta_x
    spot_price = y / (x * (1 + t))
    slippage_pct = max(0.0, ((spot_price - exec_price) / spot_price) * 100.0)
    return {"x_new": x_new, "y_new": y_new, "delta_y": delta_y, "slippage_pct": slippage_pct}

def notional_borrow(x: float, y: float, delta_x: float, T: float) -> dict:
    """Notional Finance Constant Product baseline execution"""
    k = x * y
    x_new = x + delta_x
    y_new = k / x_new
    delta_y = y - y_new

    exec_price = delta_y / delta_x
    spot_price = y / (x * (1 + T))
    slippage_pct = max(0.0, ((spot_price - exec_price) / spot_price) * 100.0)
    return {"x_new": x_new, "y_new": y_new, "delta_y": delta_y, "slippage_pct": slippage_pct}

# Quick Test Output Verification
if __name__ == "__main__":
    r = get_reference_rate(1000, 1000, 0.05, 0.5)
    C = get_invariant_C(1000, 1000, 1.0, 0.05, 0.5)
    b_res = execute_borrow(1000, 1000, 50, 1.0, 0.05, 0.5)
    print(f"✅ Rate: {r:.4f} | Invariant C: {C:.4f} | Borrow dy: {b_res['delta_y']:.4f} | Slippage: {b_res['slippage_pct']:.2f}%")

Overwriting bondmm_core.py


In [7]:
%%writefile rate_engine.py
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from prophet import Prophet

"""
rate_engine.py
Stochastic (Vasicek) and Prophet Rate Engines
"""

def generate_vasicek_rates(r0=0.05, a=0.15, b=0.05, sigma=0.015, days=180, seed=42) -> pd.DataFrame:
    """Vasicek mean-reverting stochastic interest rate generator"""
    if seed is not None:
        np.random.seed(seed)
    dt = 1.0 / 365.0
    rates = np.zeros(days)
    rates[0] = r0
    for t in range(1, days):
        dr = a * (b - rates[t - 1]) * dt + sigma * np.sqrt(dt) * np.random.normal()
        rates[t] = max(0.0001, rates[t - 1] + dr)
    start_date = datetime.now().date()
    dates = [start_date + timedelta(days=i) for i in range(days)]
    return pd.DataFrame({"ds": pd.to_datetime(dates), "rate": rates})

def generate_mock_aave_data(days=180, seed=42) -> pd.DataFrame:
    """Generates synthetic historical AAVE interest rate dataset"""
    np.random.seed(seed)
    end_date = datetime.now().date()
    dates = pd.date_range(start=end_date - timedelta(days=days), periods=days, freq="D")
    t = np.linspace(0, 4 * np.pi, days)
    y = np.clip(0.04 + 0.015 * (t / (4 * np.pi)) + 0.005 * np.sin(t) + np.random.normal(0, 0.002, days), 0.005, 0.20)
    return pd.DataFrame({"ds": dates, "y": y})

def forecast_rates_prophet(df_history: pd.DataFrame, forecast_days=30) -> pd.DataFrame:
    """Fits Prophet model on mock historical rate data and predicts future rates"""
    model = Prophet(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=False, interval_width=0.95)
    model.fit(df_history)
    future = model.make_future_dataframe(periods=forecast_days, freq="D")
    forecast = model.predict(future)
    return forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(forecast_days).reset_index(drop=True)

Overwriting rate_engine.py


In [8]:
%%writefile app.py
import streamlit as st
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import bondmm_core as bmm
import rate_engine as re

st.set_page_config(page_title="BondMM Complete Simulator", layout="wide")
st.title("⚡ BondMM Automated Market Maker & Protocol Benchmark")

# Tab Layout
tab1, tab2 = st.tabs(["🎮 Interactive Sandbox (Prompt 3)", "📊 Comparative Benchmark Dashboard (Prompt 4)"])

# =====================================================================
# TAB 1: INTERACTIVE SANDBOX (PROMPT 3)
# =====================================================================
with tab1:
    if "x" not in st.session_state:
        st.session_state.x = 1000.0
        st.session_state.y = 1000.0

    col_ctrl, col_chart = st.columns([1, 1.2])

    with col_ctrl:
        st.subheader("⚙️ Order & Pool Controls")
        r_star = st.slider("Anchor Rate (r*)", 0.01, 0.20, 0.05, step=0.005, key="sb_rstar")
        kappa = st.slider("Concentration (κ)", 0.1, 2.0, 0.5, step=0.1, key="sb_kappa")
        T_years = st.slider("Maturity (T in Years)", 0.1, 10.0, 1.0, step=0.1, key="sb_T")

        action = st.radio("Trade Type", ["Borrow (Deposit Principal Δx)", "Lend (Deposit Cash Δy)", "Cross-Swap (T1 -> T2)"], key="sb_action")
        trade_amt = st.number_input("Trade Amount", min_value=1.0, value=50.0, step=10.0, key="sb_amt")

        x_curr, y_curr = st.session_state.x, st.session_state.y
        x_post, y_post = x_curr, y_curr

        try:
            if action.startswith("Borrow"):
                res = bmm.execute_borrow(x_curr, y_curr, trade_amt, T_years, r_star, kappa)
                x_post, y_post = res["x_new"], res["y_new"]
                st.success(f"Received Cash (Δy): **{res['delta_y']:.4f}** | Slippage: **{res['slippage_pct']:.2f}%**")
            elif action.startswith("Lend"):
                res = bmm.execute_lend(x_curr, y_curr, trade_amt, T_years, r_star, kappa)
                x_post, y_post = res["x_new"], res["y_new"]
                st.success(f"Received Bonds (Δx): **{res['delta_x']:.4f}** | Slippage: **{res['slippage_pct']:.2f}%**")
            else:
                swap_res = bmm.execute_cross_swap(x_curr, y_curr, T_years, T_years + 1.0, trade_amt, r_star, kappa)
                x_post, y_post = swap_res["x1_new"], swap_res["y1_new"]
                st.success(f"Swapped into T2={T_years+1.0}y. Received Δx2: **{swap_res['final_delta_x2']:.4f}**")
        except Exception as e:
            st.error(f"Execution Error: {e}")

        # Equity Check
        eq_info = bmm.check_lp_equity(x_post, y_post, T_years)
        st.markdown("---")
        st.metric("LP Equity", f"{eq_info['current_equity']:.2f}", delta=f"{(eq_info['ratio']-1)*100:.2f}%")
        if eq_info["is_safe"]:
            st.success("✅ LP Equity Check (>90% threshold) Passed")
        else:
            st.warning("⚠️ Warning: LP Equity ratio below 90% threshold!")

        if st.button("Execute & Commit Trade", key="sb_commit"):
            st.session_state.x, st.session_state.y = x_post, y_post
            st.rerun()

    with col_chart:
        st.subheader("📈 2D Invariant Curve Plot")
        alpha = bmm.calculate_alpha(kappa)
        K = bmm.calculate_K(r_star, T_years, alpha)
        C = bmm.get_invariant_C(x_curr, y_curr, T_years, r_star, kappa)

        x_vals = np.linspace(100, max(x_curr, x_post) * 2, 200)
        y_vals = [(C - K * (xv**alpha))**(1/alpha) if (C - K * (xv**alpha)) > 0 else np.nan for xv in x_vals]

        fig_inv = go.Figure()
        fig_inv.add_trace(go.Scatter(x=x_vals, y=y_vals, mode="lines", name="Invariant C", line=dict(color="blue")))
        fig_inv.add_trace(go.Scatter(x=[x_curr], y=[y_curr], mode="markers", marker=dict(size=12, color="green"), name="Current State"))
        fig_inv.add_trace(go.Scatter(x=[x_post], y=[y_post], mode="markers", marker=dict(size=12, color="red"), name="Post-Trade State"))
        fig_inv.update_layout(xaxis_title="Principal Reserve (x)", yaxis_title="Cash Reserve (y)", height=320, margin=dict(l=10, r=10, t=30, b=10))
        st.plotly_chart(fig_inv, use_container_width=True)

        st.subheader("📊 Implied Yield Curve Plot")
        maturities = np.linspace(30/365, 10.0, 40)
        implied_rates = [bmm.get_reference_rate(x_post, y_post, r_star, kappa) * (1 + m*0.02) for m in maturities]

        fig_yield = go.Figure()
        fig_yield.add_trace(go.Scatter(x=maturities, y=implied_rates, mode="lines+markers", line=dict(color="orange"), name="Implied Rate r"))
        fig_yield.update_layout(xaxis_title="Maturity T (Years)", yaxis_title="Implied Interest Rate r", height=280, margin=dict(l=10, r=10, t=30, b=10))
        st.plotly_chart(fig_yield, use_container_width=True)


# =====================================================================
# TAB 2: COMPARATIVE BENCHMARK DASHBOARD (PROMPT 4)
# =====================================================================
with tab2:
    st.header("🏆 Comparative Benchmark: BondMM vs Yield Protocol vs Notional")
    st.markdown("Automated comparative simulation using Prophet forecasted interest rate dynamics.")

    b_col1, b_col2 = st.columns([1, 2])
    with b_col1:
        st.subheader("Simulation Parameters")
        sim_steps = st.slider("Simulation Steps (N)", min_value=10, max_value=60, value=30, step=5)
        trade_size = st.number_input("Trade Size per Step", value=30.0, step=5.0)

        if st.button("🚀 Run Comparative Simulation"):
            st.session_state.run_sim = True

    # Run Benchmark Simulation
    if st.session_state.get("run_sim", True):
        # 1. Fetch Prophet Forecasted Rates
        df_aave = re.generate_mock_aave_data(days=120)
        df_forecast = re.forecast_rates_prophet(df_aave, forecast_days=sim_steps)
        forecast_rates = df_forecast["yhat"].values

        # 2. Simulate N Consecutive Trade Steps for All 3 Protocols
        bmm_slippage_short, yield_slippage_short, notional_slippage_short = [], [], []
        bmm_slippage_long, yield_slippage_long, notional_slippage_long = [], [], []

        bmm_equity_history, yield_equity_history, notional_equity_history = [], [], []

        # State track
        x_bmm, y_bmm = 1000.0, 1000.0
        x_yp, y_yp = 1000.0, 1000.0
        x_not, y_not = 1000.0, 1000.0

        T_short = 0.25  # 3 months
        T_long = 5.0    # 5 years

        for i in range(sim_steps):
            r_curr = max(0.01, forecast_rates[i])

            # --- Short Maturity Trade (T = 0.25) ---
            bmm_res_s = bmm.execute_borrow(x_bmm, y_bmm, trade_size, T_short, r_curr, kappa=0.5)
            yp_res_s = bmm.yield_space_borrow(x_yp, y_yp, trade_size, T_short)
            not_res_s = bmm.notional_borrow(x_not, y_not, trade_size, T_short)

            bmm_slippage_short.append(bmm_res_s["slippage_pct"])
            yield_slippage_short.append(yp_res_s["slippage_pct"])
            notional_slippage_short.append(not_res_s["slippage_pct"])

            # --- Long Maturity Trade (T = 5.0) ---
            bmm_res_l = bmm.execute_borrow(x_bmm, y_bmm, trade_size, T_long, r_curr, kappa=0.5)
            yp_res_l = bmm.yield_space_borrow(x_yp, y_yp, trade_size, T_long)
            not_res_l = bmm.notional_borrow(x_not, y_not, trade_size, T_long)

            bmm_slippage_long.append(bmm_res_l["slippage_pct"])
            yield_slippage_long.append(yp_res_l["slippage_pct"])
            notional_slippage_long.append(not_res_l["slippage_pct"])

            # Update State with Short Trade
            x_bmm, y_bmm = bmm_res_s["x_new"], bmm_res_s["y_new"]
            x_yp, y_yp = yp_res_s["x_new"], yp_res_s["y_new"]
            x_not, y_not = not_res_s["x_new"], not_res_s["y_new"]

            # Record LP Equity over time
            bmm_equity_history.append(bmm.check_lp_equity(x_bmm, y_bmm, T_short)["current_equity"])
            yield_equity_history.append(y_yp + (x_yp / (1 + T_short)))
            notional_equity_history.append(y_not + (x_not / (1 + T_short)))

        # Summary Metric Cards
        st.markdown("### 📌 Executive Summary & Metric Cards")
        m1, m2, m3, m4 = st.columns(4)

        avg_bmm_slip = np.mean(bmm_slippage_long)
        avg_yp_slip = np.mean(yield_slippage_long)
        slip_reduction = ((avg_yp_slip - avg_bmm_slip) / avg_yp_slip) * 100.0

        m1.metric("BondMM Avg Slippage (Long T)", f"{avg_bmm_slip:.2f}%", f"-{slip_reduction:.1f}% vs YieldSpace")
        m2.metric("Yield Protocol Avg Slippage", f"{avg_yp_slip:.2f}%")
        m3.metric("Notional Avg Slippage", f"{np.mean(notional_slippage_long):.2f}%")
        m4.metric("BondMM Final LP Equity", f"{bmm_equity_history[-1]:.2f}", f"{(bmm_equity_history[-1]/1500 - 1)*100:.1f}%")

        st.markdown("---")
        st.subheader("📊 Slippage & Execution Impact: Short vs. Long Maturities")

        df_slip = pd.DataFrame({
            "Protocol": ["BondMM", "Yield Protocol", "Notional"] * 2,
            "Maturity": ["Short (0.25y)"] * 3 + ["Long (5.0y)"] * 3,
            "Avg Slippage (%)": [
                np.mean(bmm_slippage_short), np.mean(yield_slippage_short), np.mean(notional_slippage_short),
                np.mean(bmm_slippage_long), np.mean(yield_slippage_long), np.mean(notional_slippage_long)
            ]
        })

        fig_hist = px.bar(
            df_slip, x="Maturity", y="Avg Slippage (%)", color="Protocol", barmode="group",
            title="Slippage / Price Impact Across Short vs. Long Maturities"
        )
        st.plotly_chart(fig_hist, use_container_width=True)

        st.subheader("🛡️ LP Equity Protection Over Time")
        df_equity = pd.DataFrame({
            "Step": range(1, sim_steps + 1),
            "BondMM": bmm_equity_history,
            "Yield Protocol": yield_equity_history,
            "Notional": notional_equity_history
        })

        fig_eq = px.line(
            df_equity, x="Step", y=["BondMM", "Yield Protocol", "Notional"],
            title="LP Equity Stability During Market Rate Fluctuations",
            labels={"value": "LP Equity (E)", "variable": "Protocol"}
        )
        st.plotly_chart(fig_eq, use_container_width=True)

Overwriting app.py


In [10]:
# Launch Streamlit app directly inside Colab cell output
import time
from google.colab import output

# 1. Kill any existing instances
!pkill -f streamlit

# 2. Run Streamlit with CORS and XSRF protection disabled so Colab's iframe can embed it
!streamlit run app.py --server.port 8501 --server.enableCORS false --server.enableXsrfProtection false &> /dev/null &

# 3. Give the server a few extra seconds to fully boot up
time.sleep(5)

# 4. Display the app inside the cell
output.serve_kernel_port_as_iframe(8501, height=1000)

<IPython.core.display.Javascript object>